In [1]:
# 1. Importar librerías
from pathlib import Path
import joblib
import polars as pl
import pandas as pd
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# 2. Definir rutas
RUTA_PROYECTO = Path.cwd().parent
RUTA_DATASET = RUTA_PROYECTO / "data" / "dataset_proyecto.csv"
RUTA_MODELO = RUTA_PROYECTO / "models" / "xgboost_fraude.joblib"
RUTA_PREPROCESADOR = RUTA_PROYECTO / "models" / "preprocessor.joblib"
RUTA_REPORTES = RUTA_PROYECTO / "reports"
RUTA_REPORTES.mkdir(parents=True, exist_ok=True)

# 3. Cargar modelo, preprocesador y datos
modelo_xgb = joblib.load(RUTA_MODELO)
preprocessor = joblib.load(RUTA_PREPROCESADOR)
df = pl.read_csv(RUTA_DATASET)

print("Modelo, preprocesador y dataset cargados correctamente.")
print("Dimensiones del dataset:", df.shape)

Modelo, preprocesador y dataset cargados correctamente.
Dimensiones del dataset: (6362620, 11)


In [2]:
# 4. Seleccionar el periodo de producción
df_prod = df.filter(pl.col("step") >= 401)

FEATURES = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

X_prod = df_prod.select(FEATURES)
y_prod = df_prod["isFraud"].to_numpy()

# 5. Transformar los predictores con el preprocesador guardado
X_prod_pd = pd.DataFrame(
    X_prod.to_dict(as_series=False)
)

X_prod_transformed = preprocessor.transform(X_prod_pd)

print("Registros de producción:", X_prod_transformed.shape[0])
print("Variables transformadas:", X_prod_transformed.shape[1])
print("Fraudes reales:", int(y_prod.sum()))

Registros de producción: 575590
Variables transformadas: 11
Fraudes reales: 3736


In [3]:
# 6. Obtener probabilidades y predicciones
y_prob = modelo_xgb.predict_proba(X_prod_transformed)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# 7. Calcular métricas
pr_auc = average_precision_score(y_prod, y_prob)
roc_auc = roc_auc_score(y_prod, y_prob)

matriz = confusion_matrix(y_prod, y_pred)
tn, fp, fn, tp = matriz.ravel()

print("=== MONITORIZACIÓN DEL MODELO ===")
print(f"Average Precision (PR-AUC): {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Precisión de fraude: {tp / (tp + fp):.4f}")
print(f"Recall de fraude: {tp / (tp + fn):.4f}")
print(f"Falsas alarmas: {fp}")
print(f"Fraudes no detectados: {fn}")

print("\nMatriz de confusión:")
print(matriz)

print("\nReporte de clasificación:")
print(classification_report(
    y_prod,
    y_pred,
    target_names=["Legítimo", "Fraude"],
    digits=4,
    zero_division=0
))

=== MONITORIZACIÓN DEL MODELO ===
Average Precision (PR-AUC): 0.9726
ROC-AUC: 0.9998
Precisión de fraude: 0.5714
Recall de fraude: 0.9971
Falsas alarmas: 2794
Fraudes no detectados: 11

Matriz de confusión:
[[569060   2794]
 [    11   3725]]

Reporte de clasificación:
              precision    recall  f1-score   support

    Legítimo     1.0000    0.9951    0.9975    571854
      Fraude     0.5714    0.9971    0.7265      3736

    accuracy                         0.9951    575590
   macro avg     0.7857    0.9961    0.8620    575590
weighted avg     0.9972    0.9951    0.9958    575590



In [4]:
# 8. Crear resumen de métricas
metricas = pd.DataFrame([{
    "periodo": "Produccion (step >= 401)",
    "registros": len(y_prod),
    "fraudes_reales": int(y_prod.sum()),
    "umbral": 0.5,
    "average_precision": pr_auc,
    "roc_auc": roc_auc,
    "precision_fraude": tp / (tp + fp),
    "recall_fraude": tp / (tp + fn),
    "falsas_alarmas": int(fp),
    "fraudes_no_detectados": int(fn)
}])

# Guardar CSV
RUTA_METRICAS = RUTA_REPORTES / "metricas_produccion.csv"
metricas.to_csv(RUTA_METRICAS, index=False)

print("Métricas guardadas correctamente:")
print(RUTA_METRICAS.resolve())
print("\nResumen:")
print(metricas.round(4).to_string(index=False))

Métricas guardadas correctamente:
D:\ML_III\reports\metricas_produccion.csv

Resumen:
                 periodo  registros  fraudes_reales  umbral  average_precision  roc_auc  precision_fraude  recall_fraude  falsas_alarmas  fraudes_no_detectados
Produccion (step >= 401)     575590            3736     0.5             0.9726   0.9998            0.5714         0.9971            2794                     11


In [6]:
%pip install mlflow

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ------------ --------------------------- 3.7/12.0 MB 15.6 MB/s eta 0:00:01
   -------------------------- ------------- 7.9/12.0 MB 17.4 MB/s eta 0:00:01
   ------------------------------------- -- 11.3/12.0 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 15.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ------------------------------ --------- 3.1/4.1 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 14.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 15.6 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------------------------------------- - 3.7/3.8 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 14.3 MB/s  0:00:00
   ---------------------------------

  You can safely remove it manually.
  You can safely remove it manually.


In [7]:
import mlflow

print("Versión de MLflow:", mlflow.__version__)

Versión de MLflow: 3.17.1


In [9]:
from pathlib import Path
import mlflow

# Base de datos local para el seguimiento
RUTA_PROYECTO = Path.cwd().parent
RUTA_MLFLOW = RUTA_PROYECTO / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{RUTA_MLFLOW.as_posix()}")
mlflow.set_experiment("deteccion_fraude")

print("MLflow configurado correctamente.")
print("Base de datos:", RUTA_MLFLOW.resolve())

2026/10/18 18:49:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/18 18:49:22 INFO mlflow.store.db.utils: Updating database tables
2026/10/18 18:49:26 INFO mlflow.tracking.fluent: Experiment with name 'deteccion_fraude' does not exist. Creating a new experiment.


MLflow configurado correctamente.
Base de datos: D:\ML_III\mlflow.db


In [10]:
with mlflow.start_run(run_name="xgboost_evaluacion_temporal"):

    # Parámetros del modelo
    mlflow.log_params({
        "modelo": "XGBoost",
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": 1291.61,
        "umbral": 0.5,
        "periodo_entrenamiento": "step <= 400",
        "periodo_evaluacion": "step >= 401"
    })

    # Métricas de evaluación temporal
    mlflow.log_metrics({
        "average_precision": float(pr_auc),
        "roc_auc": float(roc_auc),
        "precision_fraude": float(tp / (tp + fp)),
        "recall_fraude": float(tp / (tp + fn)),
        "falsas_alarmas": int(fp),
        "fraudes_no_detectados": int(fn)
    })

    print("Experimento registrado correctamente.")
    print("Run ID:", mlflow.active_run().info.run_id)

Experimento registrado correctamente.
Run ID: af8422b98b4a4bebb82b6f4bfaf56440


In [11]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix
)

RUTA_PROYECTO = Path.cwd().parent
RUTA_MODELOS = RUTA_PROYECTO / "models"

# Cargar ambos modelos
modelo_original = joblib.load(
    RUTA_MODELOS / "xgboost_fraude.joblib"
)

modelo_optimizado = joblib.load(
    RUTA_MODELOS / "xgboost_fraude_optimizado.joblib"
)

# Evaluar sobre las mismas observaciones de producción
resultados = []

for nombre, modelo in [
    ("XGBoost original", modelo_original),
    ("XGBoost optimizado", modelo_optimizado)
]:
    probabilidades = modelo.predict_proba(X_prod_transformed)[:, 1]
    predicciones = (probabilidades >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_prod, predicciones, labels=[0, 1]
    ).ravel()

    resultados.append({
        "Modelo": nombre,
        "Average Precision (PR-AUC)": average_precision_score(
            y_prod, probabilidades
        ),
        "ROC-AUC": roc_auc_score(y_prod, probabilidades),
        "Precision fraude": precision_score(
            y_prod, predicciones, zero_division=0
        ),
        "Recall fraude": recall_score(
            y_prod, predicciones, zero_division=0
        ),
        "Falsas alarmas": int(fp),
        "Fraudes no detectados": int(fn)
    })

comparacion = pd.DataFrame(resultados)
display(comparacion.round(4))

,Modelo,Average Precision (PR-AUC),ROC-AUC,Precision fraude,Recall fraude,Falsas alarmas,Fraudes no detectados
0,XGBoost original,0.9726,0.9998,0.5714,0.9971,2794,11
1,XGBoost optimizado,0.9516,0.9996,0.2796,0.9997,9621,1


In [12]:
import mlflow
from pathlib import Path

RUTA_PROYECTO = Path.cwd().parent
RUTA_MLFLOW = RUTA_PROYECTO / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{RUTA_MLFLOW.as_posix()}"
)
mlflow.set_experiment("deteccion_fraude")

with mlflow.start_run(run_name="comparacion_modelos_produccion"):
    for _, fila in comparacion.iterrows():
        prefijo = (
            "original"
            if fila["Modelo"] == "XGBoost original"
            else "optimizado"
        )

        for metrica in [
            "Average Precision (PR-AUC)",
            "ROC-AUC",
            "Precision fraude",
            "Recall fraude",
            "Falsas alarmas",
            "Fraudes no detectados"
        ]:
            nombre_metrica = (
                metrica.lower()
                .replace(" ", "_")
                .replace("-", "_")
                .replace("(", "")
                .replace(")", "")
            )
            mlflow.log_metric(
                f"{prefijo}_{nombre_metrica}",
                float(fila[metrica])
            )

    mlflow.log_param("umbral_clasificacion", 0.5)
    mlflow.log_param("periodo_evaluacion", "step >= 401")

print("Comparación de modelos registrada en MLflow.")

Comparación de modelos registrada en MLflow.


In [17]:
import pandas as pd
import polars as pl
from scipy.stats import wasserstein_distance

variables_numericas = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

n_muestra = 100_000

# Muestra de referencia (pandas)
referencia = df_ref.sample(
    n=min(n_muestra, len(df_ref)),
    random_state=42
)

# Muestra de producción (Polars)
produccion = df_prod.sample(
    n=min(n_muestra, len(df_prod)),
    seed=42
).to_pandas()

resultados_drift = []

for variable in variables_numericas:
    distancia = wasserstein_distance(
        referencia[variable].to_numpy(),
        produccion[variable].to_numpy()
    )

    resultados_drift.append({
        "Variable": variable,
        "Media referencia": referencia[variable].mean(),
        "Media producción": produccion[variable].mean(),
        "Distancia Wasserstein": distancia
    })

tabla_drift = pd.DataFrame(resultados_drift)
display(tabla_drift.round(3))

,Variable,Media referencia,Media producción,Distancia Wasserstein
0,step,216.715,511.371,294.656
1,amount,180458.278,164490.955,16337.658
2,oldbalanceOrg,832182.432,723083.175,124418.963
3,newbalanceOrig,854070.175,737298.331,125095.658
4,oldbalanceDest,1100612.891,1194898.166,133349.888
5,newbalanceDest,1226825.340,1283534.822,124284.199


In [18]:
from pathlib import Path

RUTA_PROYECTO = Path.cwd().parent
RUTA_REPORTES = RUTA_PROYECTO / "reports"
RUTA_REPORTES.mkdir(exist_ok=True)

ruta_drift = RUTA_REPORTES / "data_drift_wasserstein.csv"

tabla_drift.to_csv(ruta_drift, index=False)

print("Resultados de data drift guardados en:")
print(ruta_drift)

Resultados de data drift guardados en:
d:\ML_III\reports\data_drift_wasserstein.csv


In [19]:
# Prevalencia de fraude en referencia
prevalencia_ref = df_ref["isFraud"].mean() * 100

# Prevalencia de fraude en producción
prevalencia_prod = (
    df_prod["isFraud"].mean() * 100
)

comparacion_fraude = pd.DataFrame({
    "Periodo": ["Referencia", "Producción"],
    "Total transacciones": [
        len(df_ref),
        len(df_prod)
    ],
    "Fraudes": [
        int(df_ref["isFraud"].sum()),
        int(df_prod["isFraud"].sum())
    ],
    "Prevalencia_fraude_%": [
        prevalencia_ref,
        prevalencia_prod
    ]
})

display(comparacion_fraude.round(4))

,Periodo,Total transacciones,Fraudes,Prevalencia_fraude_%
0,Referencia,5787030,4477,0.0774
1,Producción,575590,3736,0.6491


In [20]:
ruta_prevalencia = RUTA_REPORTES / "prevalencia_fraude.csv"

comparacion_fraude.to_csv(
    ruta_prevalencia,
    index=False
)

print("Comparación de prevalencia guardada en:")
print(ruta_prevalencia)

Comparación de prevalencia guardada en:
d:\ML_III\reports\prevalencia_fraude.csv


In [21]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RUTA_PROYECTO = Path.cwd().parent
RUTA_REPORTES = RUTA_PROYECTO / "reports"
RUTA_REPORTES.mkdir(exist_ok=True)

# Preparar tablas para el informe
html_drift = tabla_drift.round(3).to_html(
    index=False, border=0, classes="tabla"
)

html_prevalencia = comparacion_fraude.round(4).to_html(
    index=False, border=0, classes="tabla"
)

html_modelos = comparacion.round(4).to_html(
    index=False, border=0, classes="tabla"
)

html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Monitorización del modelo de fraude</title>
<style>
body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 35px auto;
    padding: 0 20px;
    color: #243247;
    line-height: 1.6;
}}
h1, h2 {{ color: #174a70; }}
.tabla {{
    border-collapse: collapse;
    width: 100%;
    margin: 18px 0 30px;
    font-size: 14px;
}}
.tabla th, .tabla td {{
    border: 1px solid #d9e1e8;
    padding: 10px;
    text-align: right;
}}
.tabla th {{ background: #eaf1f7; }}
.tabla th:first-child, .tabla td:first-child {{
    text-align: left;
}}
.nota {{
    background: #f3f6f9;
    padding: 14px;
    border-radius: 6px;
}}
</style>
</head>
<body>
<h1>Monitorización del modelo de detección de fraude</h1>
<p><strong>Proyecto:</strong> Detección de transacciones fraudulentas</p>
<p><strong>Referencia:</strong> step ≤ 400 |
<strong>Producción:</strong> step ≥ 401</p>

<h2>1. Comparación del rendimiento de los modelos</h2>
{html_modelos}

<h2>2. Cambio en la prevalencia del fraude</h2>
{html_prevalencia}

<h2>3. Comparación de distribuciones (data drift)</h2>
{html_drift}

<div class="nota">
<strong>Interpretación:</strong> las diferencias observadas constituyen
señales para investigar cambios en los datos. La distancia de Wasserstein
depende de la escala de cada variable y no tiene un umbral universal.
El aumento de la prevalencia del fraude no demuestra por sí solo
concept drift ni deterioro del modelo.
</div>

<p><em>Informe generado a partir de los análisis del notebook de monitorización.</em></p>
</body>
</html>
"""

ruta_html = RUTA_REPORTES / "informe_monitorizacion.html"
ruta_html.write_text(html, encoding="utf-8")

print("Informe HTML generado correctamente:")
print(ruta_html)

Informe HTML generado correctamente:
d:\ML_III\reports\informe_monitorizacion.html
